# V2 — Soil Property Modelling Report
**Konya Agricultural Plain (~23×20 km) · 10 m resolution**

Random Forest regression trained on S1 SAR + S2 optical + terrain features  
against SoilGrids v2.0 ground truth (250 m) → downscaled to 10 m predictions.

In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import rasterio

ROOT      = Path("..") if Path("../data").exists() else Path(".")
PROCESSED = ROOT / "data/processed"
INTERIM   = ROOT / "data/interim"
NODATA    = -9999.0

plt.rcParams.update({
    "figure.facecolor": "#0d1117",
    "axes.facecolor":   "#161b22",
    "axes.edgecolor":   "#30363d",
    "text.color":       "#e6edf3",
    "axes.labelcolor":  "#8b949e",
    "xtick.color":      "#8b949e",
    "ytick.color":      "#8b949e",
    "axes.titlecolor":  "#e6edf3",
    "axes.grid":        True,
    "grid.color":       "#21262d",
    "grid.linewidth":   0.5,
})

metrics = json.loads((PROCESSED / "v2_metrics.json").read_text())
print("Metrics loaded:", [m["target"] for m in metrics])

Metrics loaded: ['clay', 'sand', 'soc']


## 1 · Prediction Maps

In [2]:
def load(path, band=1):
    with rasterio.open(path) as src:
        d = src.read(band).astype("float32")
        nd = src.nodata or NODATA
    return np.ma.masked_where(d == nd, d)

def ext(path):
    with rasterio.open(path) as src:
        b = src.bounds
    return [b.left, b.right, b.bottom, b.top]

def clip(a, lo=2, hi=98):
    p = np.nanpercentile(a.compressed(), [lo, hi])
    return np.clip(a, *p)

TARGETS = [
    ("clay", "Clay Content",          "g/kg → %",   "YlOrBr", 0.1),
    ("sand", "Sand Content",          "g/kg → %",   "OrRd",   0.1),
    ("soc",  "Soil Organic Carbon",   "dg/kg",      "YlGn",   1.0),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("V2 — Soil Property Predictions · Konya 10 m · Random Forest",
             fontsize=13, fontweight="bold", y=1.02)

for ax, (prop, title, unit, cmap, scale) in zip(axes, TARGETS):
    data = load(PROCESSED / f"{prop}_10m_konya.tif") * scale
    im   = ax.imshow(clip(data), cmap=cmap, extent=ext(PROCESSED / f"{prop}_10m_konya.tif"), origin="upper")
    cb   = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label(unit, fontsize=8)
    valid = data.compressed()
    ax.set_title(f"{title}\nmean={valid.mean():.1f}  std={valid.std():.1f}  [{valid.min():.1f}–{valid.max():.1f}]",
                 fontsize=9)
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

plt.tight_layout()
plt.savefig(PROCESSED / "v2_prediction_maps.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

/var/folders/dp/m28_wwbs729bdz4cd4tz5_rc0000gn/T/ipykernel_18975/906743597.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2 · CV Performance

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle("Cross-Validation Performance  (5-fold)", fontsize=12, fontweight="bold")

colors = {"clay": "#f0a500", "sand": "#e05c5c", "soc": "#5cb85c"}

for ax, metric_key, ylabel in zip(axes,
    ["r2_mean", "rmse_mean", "mae_mean"],
    ["R²", "RMSE", "MAE"]):

    targets = [m["target"] for m in metrics]
    vals    = [m[metric_key] for m in metrics]
    errs    = [m[metric_key.replace("mean","std")] for m in metrics]
    bar_colors = [colors[t] for t in targets]

    bars = ax.bar(targets, vals, yerr=errs, color=bar_colors,
                  capsize=6, width=0.5, error_kw={"ecolor": "#8b949e", "linewidth": 1.5})
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(ylabel)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(errs)*0.1,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9, color="#e6edf3")

    if metric_key == "r2_mean":
        ax.axhline(0.5, color="#58a6ff", linestyle="--", linewidth=1, alpha=0.6)
        ax.text(2.4, 0.51, "0.5", color="#58a6ff", fontsize=8)
        ax.set_ylim(0, 0.7)

plt.tight_layout()
plt.savefig(PROCESSED / "v2_cv_performance.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

/var/folders/dp/m28_wwbs729bdz4cd4tz5_rc0000gn/T/ipykernel_18975/1341248899.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 · Feature Importance

In [4]:
FEATURE_GROUPS = {
    "vv_wet":     ("S1",     "#4493f8"),
    "vh_wet":     ("S1",     "#4493f8"),
    "vv_dry":     ("S1",     "#4493f8"),
    "vh_dry":     ("S1",     "#4493f8"),
    "nddi":       ("S1",     "#4493f8"),
    "slope":      ("Terrain","#f0a500"),
    "twi":        ("Terrain","#f0a500"),
    "curvature":  ("Terrain","#f0a500"),
    "bsi":        ("S2",     "#3fb950"),
    "clay_index": ("S2",     "#3fb950"),
    "ndvi":       ("S2",     "#3fb950"),
    "ndwi":       ("S2",     "#3fb950"),
    "iron_oxide": ("S2",     "#3fb950"),
}

FEATURE_LABELS = {
    "vv_wet": "VV wet", "vh_wet": "VH wet", "vv_dry": "VV dry", "vh_dry": "VH dry",
    "nddi": "NDDI", "slope": "Slope", "twi": "TWI", "curvature": "Curvature",
    "bsi": "BSI", "clay_index": "Clay Index", "ndvi": "NDVI",
    "ndwi": "NDWI", "iron_oxide": "Iron Oxide",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Feature Importances — Random Forest", fontsize=13, fontweight="bold")

for ax, m in zip(axes, metrics):
    imps  = m["feature_importances"]
    items = sorted(imps.items(), key=lambda x: x[1])
    names  = [FEATURE_LABELS[k] for k, _ in items]
    values = [v for _, v in items]
    bar_colors = [FEATURE_GROUPS[k][1] for k, _ in items]

    bars = ax.barh(names, values, color=bar_colors, height=0.65)
    ax.set_xlabel("Importance", fontsize=9)
    ax.set_title(m["target"].upper(), fontsize=11, fontweight="bold",
                 color=colors[m["target"]])

    for bar, val in zip(bars, values):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=7.5, color="#8b949e")

    ax.set_xlim(0, max(values) * 1.25)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#4493f8", label="S1 SAR"),
    Patch(facecolor="#f0a500", label="Terrain / Hydrology"),
    Patch(facecolor="#3fb950", label="S2 Optical"),
]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           framealpha=0.2, fontsize=9, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.savefig(PROCESSED / "v2_feature_importance.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

/var/folders/dp/m28_wwbs729bdz4cd4tz5_rc0000gn/T/ipykernel_18975/3321978028.py:57: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4 · Prediction Distributions

In [5]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Prediction Distributions vs SoilGrids Ground Truth", fontsize=12, fontweight="bold")

sg_dir = INTERIM / "soilgrids"

for ax, (prop, title, unit, cmap, scale) in zip(axes, TARGETS):
    color = colors[prop]

    # RF prediction
    pred = load(PROCESSED / f"{prop}_10m_konya.tif").compressed() * scale
    ax.hist(pred, bins=60, alpha=0.75, color=color, label="RF prediction (10 m)",
            density=True, edgecolor="none")

    # SoilGrids ground truth
    sg_path = sg_dir / f"sg_{prop}_0_5cm_mean.tif"
    if sg_path.exists():
        sg = load(sg_path).compressed() * scale
        ax.hist(sg, bins=30, alpha=0.5, color="#8b949e", label="SoilGrids 250 m",
                density=True, edgecolor="none")

    ax.axvline(pred.mean(), color=color, linestyle="--", linewidth=1.5,
               label=f"RF mean={pred.mean():.1f}")
    ax.set_xlabel(f"{title} ({unit})", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7.5, framealpha=0.3)

plt.tight_layout()
plt.savefig(PROCESSED / "v2_distributions.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()

/var/folders/dp/m28_wwbs729bdz4cd4tz5_rc0000gn/T/ipykernel_18975/2692649713.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5 · Key Findings

In [6]:
print("=" * 60)
print("V2 KEY FINDINGS — Konya Agricultural Plain")
print("=" * 60)

print("\n── SOIL COMPOSITION ─────────────────────────────────")
for prop, unit, scale in [("clay","g/kg",1.0),("sand","g/kg",1.0),("soc","dg/kg",1.0)]:
    d = load(PROCESSED / f"{prop}_10m_konya.tif").compressed()
    print(f"  {prop:5s}: mean={d.mean():.1f}  std={d.std():.1f}  [{d.min():.1f}–{d.max():.1f}] {unit}")

print("\n── MODEL PERFORMANCE (5-fold CV) ─────────────────────")
print(f"  {'Target':<6}  R²     RMSE    MAE")
for m in metrics:
    print(f"  {m['target']:<6}  {m['r2_mean']:.3f}  {m['rmse_mean']:>6.2f}  {m['mae_mean']:>6.2f}")

print("\n── TOP FEATURES ──────────────────────────────────────")
for m in metrics:
    top3 = sorted(m['feature_importances'].items(), key=lambda x: -x[1])[:3]
    print(f"  {m['target']:5s}: " + "  >".join(f"{k}({v:.3f})" for k,v in top3))

print("\n── INTERPRETATION ────────────────────────────────────")
print("  Clay ~37.7% → heavy clay lacustrine plain (ancient lake sediment)")
print("  Sand ~16.7% → consistent with clay-dominated geology")
print("  Iron Oxide dominant for clay → reddish/ferric soils correlate with clay")
print("  SOC R²=0.23 → organic C too spatially variable for SAR+terrain alone")
print("  TWI important for SOC → water accumulation drives organic matter")

V2 KEY FINDINGS — Konya Agricultural Plain

── SOIL COMPOSITION ─────────────────────────────────
  clay : mean=376.8  std=12.8  [319.0–418.2] g/kg
  sand : mean=166.6  std=6.9  [133.6–214.0] g/kg


  soc  : mean=193.2  std=11.9  [143.5–263.9] dg/kg

── MODEL PERFORMANCE (5-fold CV) ─────────────────────
  Target  R²     RMSE    MAE
  clay    0.485   13.98   10.87
  sand    0.430    8.28    6.30
  soc     0.229   29.42   23.52

── TOP FEATURES ──────────────────────────────────────
  clay : iron_oxide(0.220)  >bsi(0.154)  >slope(0.108)
  sand : bsi(0.193)  >iron_oxide(0.141)  >slope(0.126)
  soc  : vv_wet(0.113)  >twi(0.112)  >bsi(0.091)

── INTERPRETATION ────────────────────────────────────
  Clay ~37.7% → heavy clay lacustrine plain (ancient lake sediment)
  Sand ~16.7% → consistent with clay-dominated geology
  Iron Oxide dominant for clay → reddish/ferric soils correlate with clay
  SOC R²=0.23 → organic C too spatially variable for SAR+terrain alone
  TWI important for SOC → water accumulation drives organic matter
